# Criando o dataframe municipal de CVLI, 2022–2025

Uma linha por município cearense. O target é `log(cvli_2025) - log(cvli_2022)`. As características socioeconômicas são observadas em 2022; crescimento populacional usa as populações de 2022 e 2025. O CSV gerado substitui `crime_violento.xlsx` como base de modelagem.


## Fontes e decisões

- CVLI: `data/CVLI_2009-a-2025.xlsx`, aba `CVLI`; contagens anuais de registros por município.
- População inicial: Censo 2022 (`data/pop_2022.xlsx`); população final: estimativa IBGE 2025 (`data/tabela 6579 população residente estimada.xlsx`).
- Urbanização: população urbana do Censo 2022 (`data/população urbana (tabela 9923).xlsx`) dividida pela população total de 2022.
- Desocupação: desocupados / força de trabalho em 2022, em pontos percentuais, da tabela 9517.
- Rendimento: valor nominal médio mensal de todos os trabalhos das pessoas ocupadas de 14+ anos com rendimento, em R$ de 2022, tabela 10280.
- Densidade demográfica: habitantes por km² no Censo 2022, tabela 4714.
- Escolaridade: a planilha local da tabela 10061 traz **“Sem instrução e fundamental incompleto”**, categoria combinada entre adultos de 18+ anos. Ela não permite isolar “sem instrução”. A coluna é identificada como proxy e não deve ser interpretada como taxa somente de pessoas sem instrução.

A diferença de logs é indefinida quando alguma contagem é zero. Nesses municípios o target fica vazio; não se adiciona pseudocontagem. `df_modelo` contém apenas observações com target definido.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA = PROJECT_ROOT / 'data'
OUTPUT = PROJECT_ROOT / 'data' / 'crime_violento_2022_2025.csv'

from src.data.clean_data import clean_column_names, normalize_municipality_name, prepare_planning_regions
from src.data.load_data import load_cvli_data, load_planning_regions


## Carregar e padronizar as fontes

Todas as junções usam o código IBGE do cadastro de municípios. Nomes são normalizados somente para localizar esse código. Cada planilha tem 184 linhas municipais na aba `Tabela`.


In [2]:
cadastro = prepare_planning_regions(load_planning_regions())
assert len(cadastro) == 184 and cadastro.code_muni.is_unique
chaves = cadastro[['code_muni', 'municipio', 'municipio_key']].copy()

def ler_indicador(arquivo, primeira_linha_dados, coluna_nome, coluna_valor, nome_saida):
    bruto = pd.read_excel(
        DATA / arquivo, sheet_name='Tabela', header=None,
        skiprows=primeira_linha_dados - 1, nrows=184,
    )
    parcial = bruto.iloc[:, [coluna_nome, coluna_valor]].copy()
    parcial.columns = ['nome_fonte', nome_saida]
    parcial['municipio_key'] = (parcial.nome_fonte.astype(str)
        .str.replace(r'\s*\(CE\)\s*$', '', regex=True)
        .map(normalize_municipality_name))
    parcial[nome_saida] = pd.to_numeric(parcial[nome_saida], errors='coerce')
    assert parcial.municipio_key.is_unique, f'Chaves repetidas: {arquivo}'
    resultado = chaves[['code_muni', 'municipio_key']].merge(
        parcial[['municipio_key', nome_saida]], on='municipio_key',
        how='left', validate='one_to_one', indicator=True)
    assert resultado._merge.eq('both').all(), f'Municípios não encontrados: {arquivo}'
    assert resultado[nome_saida].notna().all(), f'Valores ausentes: {arquivo}'
    return resultado[['code_muni', nome_saida]]


In [3]:
pop_2022 = ler_indicador('pop_2022.xlsx', 6, 0, 3, 'populacao_2022')
pop_2025 = ler_indicador('tabela 6579 população residente estimada.xlsx', 3, 0, 14, 'populacao_2025')
urbana = ler_indicador('população urbana (tabela 9923).xlsx', 3, 0, 1, 'populacao_urbana_2022')
desocupacao = ler_indicador('ocupações e desocupações dentro da força de trabalho (tabela 9517).xlsx', 3, 0, 4, 'taxa_desocupacao_2022_pct')
rendimento = ler_indicador('Valor do rendimento mensal médio de todos os trabalhos (tabela 10280).xlsx', 6, 0, 2, 'rendimento_medio_mensal_2022_brl')
densidade = ler_indicador('densidade demográfica (tabela 4714).xlsx', 3, 0, 1, 'densidade_2022_hab_km2')
escolaridade = ler_indicador('pessoas adultas sem instrução (tabela 10061).xlsx', 3, 0, 1, 'taxa_sem_instrucao_ou_fundamental_incompleto_2022_pct')


## Contar CVLI nos anos inicial e final

O cadastro define os 184 municípios, inclusive os que não tiveram registro em algum ano. Esses valores são zero, não ausentes.


In [4]:
cvli = clean_column_names(load_cvli_data(sheet_name='CVLI'))
cvli['data'] = pd.to_datetime(cvli['data'], errors='coerce')
assert cvli.data.notna().all(), 'Datas de CVLI inválidas'
cvli['municipio_key'] = cvli['município'].map(normalize_municipality_name)
assert cvli.municipio_key.isin(chaves.municipio_key).all(), 'Município de CVLI fora do cadastro'
contagens = (cvli.loc[cvli.data.dt.year.isin([2022, 2025])]
    .assign(ano=lambda d: d.data.dt.year)
    .groupby(['municipio_key', 'ano']).size()
    .unstack(fill_value=0).reindex(index=chaves.municipio_key, columns=[2022, 2025], fill_value=0))
contagens.columns = ['cvli_2022', 'cvli_2025']
contagens = chaves[['code_muni', 'municipio_key']].merge(
    contagens.reset_index(), on='municipio_key', validate='one_to_one')
assert contagens[['cvli_2022', 'cvli_2025']].ge(0).all().all()


## Montar o dataframe e calcular as variáveis

`crescimento_populacional_2022_2025_pct` é uma variação observada ao longo do mesmo intervalo do target. Portanto, ela não é uma característica conhecida no início de 2022.


In [5]:
df = chaves[['code_muni', 'municipio']].copy()
for tabela in [contagens, pop_2022, pop_2025, urbana, desocupacao, rendimento, densidade, escolaridade]:
    df = df.merge(tabela.drop(columns='municipio_key', errors='ignore'),
                  on='code_muni', how='left', validate='one_to_one')

df['urbanizacao_2022'] = df.populacao_urbana_2022 / df.populacao_2022
df['crescimento_populacional_2022_2025_pct'] = (
    (df.populacao_2025 / df.populacao_2022) - 1) * 100
df['crescimento_cvli_log_2022_2025'] = np.where(
    df.cvli_2022.gt(0) & df.cvli_2025.gt(0),
    np.log(df.cvli_2025.replace(0, np.nan)) - np.log(df.cvli_2022.replace(0, np.nan)),
    np.nan,
)

colunas = [
    'code_muni', 'municipio', 'cvli_2022', 'cvli_2025',
    'crescimento_cvli_log_2022_2025', 'urbanizacao_2022',
    'taxa_desocupacao_2022_pct', 'rendimento_medio_mensal_2022_brl',
    'populacao_2022', 'crescimento_populacional_2022_2025_pct',
    'densidade_2022_hab_km2',
    'taxa_sem_instrucao_ou_fundamental_incompleto_2022_pct',
    'populacao_2025', 'populacao_urbana_2022',
]
df = df[colunas].sort_values('code_muni').reset_index(drop=True)
df_modelo = df.dropna(subset=['crescimento_cvli_log_2022_2025']).copy()


## Conferir cobertura

A tabela preserva os 184 municipios e as contagens originais. A amostra analitica com logs validos fica em `df_modelo` na memoria. A exportacao ocorre depois da inclusao de emprego.


In [ ]:
assert len(df) == 184 and df.code_muni.is_unique
assert df.drop(columns=['crescimento_cvli_log_2022_2025']).notna().all().all()
assert df.populacao_2022.gt(0).all() and df.populacao_2025.gt(0).all()
assert df.urbanizacao_2022.between(0, 1).all()
assert df.taxa_desocupacao_2022_pct.between(0, 100).all()
assert df.taxa_sem_instrucao_ou_fundamental_incompleto_2022_pct.between(0, 100).all()
assert np.allclose(df_modelo.crescimento_cvli_log_2022_2025,
                   np.log(df_modelo.cvli_2025 / df_modelo.cvli_2022))
assert len(df_modelo) + df.crescimento_cvli_log_2022_2025.isna().sum() == 184

resumo = pd.DataFrame({
    'medida': ['Municípios', 'Target definido', 'CVLI zero em 2022', 'CVLI zero em 2025',
               'CVLI estadual em 2022', 'CVLI estadual em 2025'],
    'valor': [len(df), len(df_modelo), df.cvli_2022.eq(0).sum(),
              df.cvli_2025.eq(0).sum(), df.cvli_2022.sum(), df.cvli_2025.sum()],
})
display(resumo)
display(df.head(10))


## Uso

Para regressão com o target especificado, use `df_modelo` e as sete colunas explicativas exibidas acima. A proxy de escolaridade precisa ser substituída por uma fonte com a categoria isolada “sem instrução” caso essa seja uma exigência do estudo.


## Incluir emprego setorial no CSV de modelagem

A fonte do IPECE nao informa o ano de referencia. Cada municipio tem uma linha no CSV. Contagens vazias da fonte permanecem vazias. A soma das seis taxas e considerada adequada quando e maior que 0,9.


In [14]:
from shutil import copyfile
import subprocess
from tempfile import NamedTemporaryFile

ARQUIVO_EMPREGOS = DATA / 'empregos_setores_ipece.xlsx'

# Copiar permite ler a fonte mesmo quando outro processo impede a leitura direta.
with NamedTemporaryFile(suffix='.xlsx', dir=DATA, delete=False) as temp:
    copia_empregos = Path(temp.name)
try:
    try:
        copyfile(ARQUIVO_EMPREGOS, copia_empregos)
    except PermissionError:
        # O OneDrive pode bloquear a abertura Python, embora Copy-Item consiga copiar.
        origem = str(ARQUIVO_EMPREGOS).replace("'", "''")
        destino = str(copia_empregos).replace("'", "''")
        comando = f"Copy-Item -LiteralPath '{origem}' -Destination '{destino}' -Force"
        subprocess.run(['powershell', '-NoProfile', '-Command', comando], check=True)
    bruto_empregos = pd.read_excel(copia_empregos, sheet_name='Sheet 1')
finally:
    copia_empregos.unlink(missing_ok=True)

colunas_taxas = ['tx_adm_pub', 'tx_agro', 'tx_comercio', 'tx_construcao_civil',
                 'tx_industria_transformacao', 'tx_servicos']
colunas_contagens = ['empregos_formais', 'empregos_adm_pub', 'empregos_agro',
                    'empregos_comercio', 'empregos_construcao_civil',
                    'empregos_industria_transformacao', 'empregos_servicos']
assert len(bruto_empregos) == 184
assert list(bruto_empregos.columns[8:14]) == colunas_taxas
empregos = bruto_empregos.iloc[:, :14].copy()
empregos.columns = ['municipio_fonte'] + colunas_contagens + colunas_taxas
empregos['municipio_key'] = empregos.municipio_fonte.map(normalize_municipality_name)
assert empregos.municipio_key.is_unique and empregos.municipio_key.notna().all()
for coluna in colunas_contagens + colunas_taxas:
    empregos[coluna] = pd.to_numeric(empregos[coluna], errors='coerce')
assert empregos.empregos_formais.gt(0).all()
assert empregos[colunas_taxas].notna().all().all()
assert empregos[colunas_taxas].ge(0).all().all()
empregos['soma_taxas_emprego'] = empregos[colunas_taxas].sum(axis=1)
empregos['taxas_emprego_ok'] = empregos.soma_taxas_emprego.gt(0.9)
assert empregos.soma_taxas_emprego.le(1 + 1e-9).all()
print(f'Soma > 0,9: {empregos.taxas_emprego_ok.sum()} de {len(empregos)} municipios')
display(empregos.loc[~empregos.taxas_emprego_ok,
                    ['municipio_fonte', 'empregos_formais', 'soma_taxas_emprego']])


Soma > 0,9: 182 de 184 municipios
    municipio_fonte  empregos_formais  soma_taxas_emprego
1           Acarape              1639            0.888957
180          Uruoca              1570            0.724204


In [15]:
# Juntar uma linha de emprego a cada municipio do dataframe municipal.
df['municipio_key'] = df.municipio.map(normalize_municipality_name)
novas_colunas = colunas_contagens + colunas_taxas + ['soma_taxas_emprego', 'taxas_emprego_ok']
df = df.drop(columns=novas_colunas, errors='ignore')
df = df.merge(empregos[['municipio_key'] + novas_colunas], on='municipio_key',
              how='left', validate='one_to_one', indicator=True)
assert df._merge.eq('both').all(), 'Municipios sem correspondencia'
assert len(df) == 184 and df.code_muni.is_unique
assert df.empregos_formais.notna().all()
assert df[colunas_taxas + ['soma_taxas_emprego', 'taxas_emprego_ok']].notna().all().all()
df = df.drop(columns=['municipio_key', '_merge'])
df_modelo = df.dropna(subset=['crescimento_cvli_log_2022_2025']).copy()
df.to_csv(OUTPUT, index=False, encoding='utf-8-sig')
print(f'CSV atualizado: {OUTPUT} ({len(df)} municipios, {len(df.columns)} colunas)')


CSV atualizado: C:\Users\emanu\OneDrive\Documentos\violência\data\crime_violento_2022_2025.csv (184 municipios, 29 colunas)
